# LLM-as-a-Judge

Evaluates the final summaries produced by `main.py` runs.  
Each `.log` file is parsed for its `--- SUMMARY ---` section, then judged by an LLM using criteria you define below.

In [1]:
from dotenv import load_dotenv
load_dotenv()

import glob
import re
from IPython.display import Markdown, display

from agent_chat.agents import Agent
from agent_chat.conversation import Message, call_agent

## 1. Configure the judge

Edit `JUDGE_CRITERIA` to set your evaluation criteria.  
The judge will score and comment on each plan according to what you write here.

In [2]:
JUDGE_CRITERIA = """
You are an expert evaluator of software project plans produced in agile planning meetings.

Evaluate the submitted plan on the following criteria. For each criterion give a score from 1 (poor) to 5 (excellent) and a brief justification.

Criteria:
1. Clarity — Are the user stories clearly written and unambiguous?
2. Completeness — Does the plan cover the full scope of the problem?
3. Feasibility — Are the stories realistically achievable in a sprint?
4. Testability — Can each story be verified with a concrete acceptance criterion?
5. Risk coverage — Are significant risks or unknowns acknowledged?

End your evaluation with an overall verdict (APPROVED, NEEDS REVISION, or REJECTED) and a
total score on the last line in exactly this format:
Total score: X / 25
"""

# Model and provider for the judge — change to any supported provider/model
JUDGE_MODEL = "claude-sonnet-4-6"
JUDGE_PROVIDER = "anthropic"

## 2. Load summaries from log files

In [3]:
def extract_summary(log_path: str) -> str | None:
    """Return the text after '--- SUMMARY ---' in a log file, or None if absent."""
    with open(log_path) as f:
        content = f.read()
    match = re.search(r"--- SUMMARY ---\n(.+)", content, re.DOTALL)
    return match.group(1).strip() if match else None


def extract_config(log_path: str) -> list[dict]:
    """Parse the --- CONFIG --- block into a list of agent dicts."""
    with open(log_path) as f:
        content = f.read()
    block = re.search(r"--- CONFIG ---\n(.+?)--- END CONFIG ---", content, re.DOTALL)
    if not block:
        return []
    agents = []
    for line in block.group(1).strip().splitlines():
        m = re.match(r"(\w+): model=(\S+) provider=(\S+) max_tokens=(\d+)", line)
        if m:
            agents.append({"name": m.group(1), "model": m.group(2),
                           "provider": m.group(3), "max_tokens": int(m.group(4))})
    return agents


log_files = sorted(glob.glob("logs/conversation_*.log"))

if not log_files:
    print("No logs/conversation_*.log files found. Run main.py first.")
else:
    summaries = {path: extract_summary(path) for path in log_files}
    configs   = {path: extract_config(path)   for path in log_files}
    for path in log_files:
        summary = summaries[path]
        cfg = configs[path]
        status  = f"{len(summary)} chars" if summary else "no summary found"
        agents_str = ", ".join(f"{a['name']}({a['model']})" for a in cfg) or "no config found"
        print(f"  {path}  →  {status}  |  {agents_str}")

  logs/conversation_20260305_214022.log  →  2597 chars  |  planner(Llama-4-Scout-17B-16E-Instruct), critic(Llama-4-Scout-17B-16E-Instruct)
  logs/conversation_20260305_214207.log  →  1962 chars  |  planner(claude-sonnet-4-6), critic(Llama-4-Scout-17B-16E-Instruct)
  logs/conversation_20260305_220323.log  →  4520 chars  |  planner(claude-sonnet-4-6), critic(Llama-4-Scout-17B-16E-Instruct)


## 3. Run the judge

Each summary is sent to the judge as a separate call. Results are printed below.

In [4]:
judge = Agent(
    name="judge",
    role=JUDGE_CRITERIA.strip(),
    model=JUDGE_MODEL,
    provider=JUDGE_PROVIDER,
)

verdicts: dict[str, str] = {}

for path, summary in summaries.items():
    if not summary:
        print(f"Skipping {path} — no summary section found.")
        continue

    print(f"Judging {path} ...", end=" ", flush=True)
    history = [Message(
        speaker="user",
        content=f"Here is the plan to evaluate:\n\n{summary}",
    )]
    verdict = call_agent(judge, history, system=judge.role)
    verdicts[path] = verdict
    print("done")

Judging logs/conversation_20260305_214022.log ... done
Judging logs/conversation_20260305_214207.log ... done
Judging logs/conversation_20260305_220323.log ... done


## 4. Results

In [5]:
for path, verdict in verdicts.items():
    cfg = configs.get(path, [])
    config_lines = "\n".join(
        f"- **{a['name']}**: `{a['model']}` ({a['provider']})"
        for a in cfg
    ) or "_no config recorded_"
    display(Markdown(f"---\n### {path}\n\n**Agents**\n\n{config_lines}\n\n{verdict}"))

---
### logs/conversation_20260305_214022.log

**Agents**

- **planner**: `Llama-4-Scout-17B-16E-Instruct` (azure_ai)
- **critic**: `Llama-4-Scout-17B-16E-Instruct` (azure_ai)

## Evaluation

### 1. Clarity — Score: 3/5
The stories are generally understandable and follow a consistent format with titles, descriptions, and effort points. However, several stories lack the standard "As a [user], I want [goal], so that [reason]" format, making them read more like technical tasks than user stories. Key ambiguities exist: "real-time" updating (Story 1) is vague for a command-line app, "current input" vs "result" display is never precisely defined, and Stories 3 and 6 appear to overlap significantly in responsibility. The target user and context (CLI app) could be stated more explicitly upfront.

### 2. Completeness — Score: 3/5
Core calculator functionality is reasonably covered. However, notable gaps exist: decimal/floating-point number support is never mentioned (can users enter 3.14?), multi-digit number input is not explicitly addressed (Story 2 only mentions 0–9 individually), negative number input is absent, and there is no story covering the initial/startup state of the application. The relationship between Stories 3 and 6 creates an architectural gap — it's unclear where operator *acceptance* ends and *handling* begins.

### 3. Feasibility — Score: 4/5
Effort estimates are reasonable and the total (24 points) is appropriate for a sprint focused on an MVP CLI calculator. No single story is egregiously large. Stories 5 and 6 have a dependency relationship that could cause blocking if not sequenced carefully, but this is manageable. Story 10 (left-to-right evaluation) may be harder than its effort of 3 suggests if it must integrate with Stories 5 and 6 simultaneously.

### 4. Testability — Score: 2/5
This is the plan's most significant weakness. **No story includes explicit acceptance criteria.** While some behaviors are implied (e.g., "displaying 'Error'" in Stories 7–9), none are formalized as verifiable conditions. There is no definition of done, no example inputs/outputs, and no boundary conditions stated. It would be difficult for a QA engineer to write tests directly from these stories without significant clarification.

### 5. Risk coverage — Score: 2/5
A few edge cases are addressed (division by zero, consecutive operators, non-numeric input), which is positive. However, significant risks are unacknowledged: the overlap/dependency between Stories 3 and 6 is a design risk, operator precedence is deliberately excluded with no note about user expectation management, there is no mention of state management complexity for chained operations, and the `eval()` prohibition in Story 6 is noted but its architectural implications are not discussed.

---

## Overall Verdict

The plan covers the basic functional skeleton of an MVP calculator and is feasible in scope. However, the **complete absence of acceptance criteria** is a critical flaw that makes the plan untestable as written. Story overlap and missing edge cases (decimals, multi-digit numbers) also need addressing before development begins.

**Verdict: NEEDS REVISION**

Total score: 14 / 25

---
### logs/conversation_20260305_214207.log

**Agents**

- **planner**: `claude-sonnet-4-6` (anthropic)
- **critic**: `Llama-4-Scout-17B-16E-Instruct` (azure_ai)

# Evaluation of CLI Calculator MVP Plan

## Criterion 1: Clarity — Score: 4 / 5

The stories are generally well-written and unambiguous. Titles are concise, descriptions explain both the happy path and edge cases in plain language. The decisions section helpfully resolves common ambiguities (leading zeros, chaining). Minor deductions:
- US4 says "evaluate pending operation left-to-right using stored number, operator, and **current input**" — it's slightly unclear whether the result itself becomes the new accumulator for chained operations, or if this is always a two-operand evaluation. A concrete example (e.g. `3 + 4 = → 7`, then `+ 2 = → 9`) would eliminate doubt.
- US3 and US5 have overlapping responsibilities (invalid operator position is mentioned in US3 but also arguably belongs to US5), which could cause confusion during implementation.

---

## Criterion 2: Completeness — Score: 3 / 5

Core flows are present, but several gaps exist:

- **Negative number input** is never addressed. Can a user enter `-5` as a first operand? The token-by-token model makes this genuinely ambiguous and it is not acknowledged.
- **Decimal/float input** is explicitly excluded by "integer" wording in US2, but this is never stated as an out-of-scope decision, leaving implementers uncertain about whether `3.5` should produce `Error` or be partially accepted.
- **Result chaining** (using the result of `=` as the left operand of the next operation) is mentioned only obliquely in the decisions table. It deserves its own story or explicit acceptance criterion, because it requires non-trivial state management.
- **Overflow or very large numbers** — absent, though defensible for an MVP.

---

## Criterion 3: Feasibility — Score: 5 / 5

All stories are appropriately scoped for a single sprint. Effort estimates (Low/Medium) are believable and consistent. The "No external libraries" constraint is acknowledged upfront and does not materially affect feasibility. The sequential dependency (US1 as skeleton, others building on it) is logical and supports incremental delivery. No story appears to carry hidden complexity that would cause it to blow up in implementation.

---

## Criterion 4: Testability — Score: 3 / 5

Some stories imply testable behavior but lack explicit acceptance criteria:

- US1 ("echo it back") is vague — echo *what* exactly? What is the expected output format?
- US4 does not specify the displayed format of the result (integer vs. float, e.g. does `7 / 2` display `3` or `3.5`?). This will cause test disagreements.
- US6 and US7 are straightforward and implicitly testable.
- US5 lists error conditions as a bullet list but does not specify whether *all* must pass to accept the story or if partial coverage suffices.

A simple "Given / When / Then" structure on even two or three stories would significantly strengthen this section.

---

## Criterion 5: Risk Coverage — Score: 2 / 5

This is the weakest area of the plan:

- **No risks are explicitly called out.** There is no risk register or even an informal note.
- The **result type ambiguity** (integer vs. float for division) is a hidden risk that could cause rework — e.g. `7 / 2 = 3` vs `3.5`.
- **State machine complexity** is not flagged. The interaction between US3, US4, US5, and chaining implies a non-trivial state machine that could introduce bugs at integration time.
- **Negative numbers** (noted above) are an unacknowledged unknown that could surface mid-sprint.
- The "No `eval()`" constraint in US4 is good defensive thinking but the rationale (security? learning exercise?) is not stated, which matters if scope pressure arises.

---

## Summary

| Criterion | Score |
|-----------|-------|
| Clarity | 4 / 5 |
| Completeness | 3 / 5 |
| Feasibility | 5 / 5 |
| Testability | 3 / 5 |
| Risk Coverage | 2 / 5 |

**Verdict: NEEDS REVISION**
The plan has a solid, implementable core and good feasibility. However, it requires explicit acceptance criteria on key stories, resolution of the integer/float output ambiguity, clarification of result-chaining behavior, and at minimum a brief risk/unknowns section before it is sprint-ready.

Total score: 17 / 25

---
### logs/conversation_20260305_220323.log

**Agents**

- **planner**: `claude-sonnet-4-6` (anthropic)
- **critic**: `Llama-4-Scout-17B-16E-Instruct` (azure_ai)

## Evaluation

### 1. Clarity — 5/5
The stories are well-structured and consistently formatted. Each follows the standard "As a… I want… so that…" pattern with meaningful rationale. The acceptance criteria are concrete and use unambiguous language (e.g., "Positive balance = owed money; negative balance = owes money"). The scope constraints stated upfront (single currency, no auth, ephemeral sessions) proactively eliminate a large class of ambiguity that typically plagues expense-splitter projects. Minor note: Story 2 mentions "updates without requiring a page reload" — this implies AJAX/dynamic behavior that isn't explicitly justified by the Plain HTML/JS stack choice, but it's a small inconsistency rather than a true ambiguity.

### 2. Completeness — 4/5
The plan covers the critical path thoroughly: session setup → expense entry → balance calculation → settlement display → validation. The agreed scope is appropriately constrained for a prototype. Two minor gaps worth flagging: (1) there is no story for **deleting or editing an expense**, which is almost always needed once real users start testing; (2) there is no handling of the **session end/reset** flow — how does a user start a fresh session if they want to? These are reasonable deferrals for a prototype but should be noted as known omissions.

### 3. Feasibility — 5/5
At 14 total points with a sensible mix of small and medium stories, this is a realistic sprint load for a small team (2–3 developers). No story is oversized. The recommended build order is logical and dependency-aware — decoupling the calculation engine (Story 3) from the UI and testing it in isolation is sound engineering practice. The greedy algorithm in Story 4 is a well-understood approach and not a research problem. The chosen stack (Flask + SQLite + plain HTML/JS) is appropriately lightweight for the scope.

### 4. Testability — 4/5
Most acceptance criteria are directly testable. The balance accuracy criterion ("accurate to two decimal places") is quantifiable. The N-1 transaction bound in Story 4 is a clean algorithmic invariant that can be unit tested. The validation criteria in Story 5 enumerate specific rejection cases. The minor deduction: Story 2's "updates without requiring a page reload" criterion lacks a precise definition of *how* this is verified in testing (e.g., no full HTTP request is fired? DOM updates within X ms?). Story 6 could also benefit from specifying a maximum or reasonable limit on participant count to bound edge-case testing.

### 5. Risk Coverage — 3/5
This is the weakest area. The plan does not explicitly acknowledge several meaningful risks:
- **Floating-point arithmetic**: Splitting expenses evenly across N people often produces repeating decimals. How rounding errors are handled (e.g., who absorbs the penny) is unspecified and can silently break the balance invariant (sum of all balances = 0).
- **Greedy algorithm correctness**: The N-1 bound claimed in Story 4 is not always achievable with a greedy approach on arbitrary net balances — it requires the optimal algorithm. This could be a sprint-ending surprise.
- **Session lifecycle ambiguity**: With SQLite and no authentication, concurrent or abandoned sessions could accumulate. No cleanup strategy is mentioned.
- **Scalability of "no page reload" requirement**: For a plain HTML/JS stack, this implies fetch/XHR calls that add complexity not budgeted in Story 2's 1-point estimate.

These aren't blockers, but a strong plan would name them explicitly.

---

## Overall Verdict

The plan is well-written, appropriately scoped, and immediately actionable. The build order is sensible. The primary improvement needed is explicit risk acknowledgment — particularly around floating-point rounding and the transaction-minimization algorithm's correctness guarantee. A small revision pass to address those two technical risks and the missing session-reset story would make this plan production-ready.

**Verdict: NEEDS REVISION**

Total score: 21 / 25

## 5. Leaderboard

In [6]:
def extract_score(verdict: str) -> int | None:
    """Parse 'Total score: X / Y' from a verdict string."""
    m = re.search(r"Total score:\s*(\d+)\s*/\s*\d+", verdict, re.IGNORECASE)
    return int(m.group(1)) if m else None


rows = []
for path, verdict in verdicts.items():
    cfg = configs.get(path, [])
    score = extract_score(verdict)
    models = " / ".join(f"{a['name']}={a['model']}" for a in cfg) or "unknown"
    rows.append({"run": path, "models": models, "score": score})

rows.sort(key=lambda r: (r["score"] or -1), reverse=True)

header = "| Rank | Run | Models | Score |\n|------|-----|--------|-------|\n"
body = "\n".join(
    f"| {i+1} | {r['run']} | {r['models']} | {r['score'] if r['score'] is not None else '—'} |"
    for i, r in enumerate(rows)
)
display(Markdown(header + body))

| Rank | Run | Models | Score |
|------|-----|--------|-------|
| 1 | logs/conversation_20260305_220323.log | planner=claude-sonnet-4-6 / critic=Llama-4-Scout-17B-16E-Instruct | 21 |
| 2 | logs/conversation_20260305_214207.log | planner=claude-sonnet-4-6 / critic=Llama-4-Scout-17B-16E-Instruct | 17 |
| 3 | logs/conversation_20260305_214022.log | planner=Llama-4-Scout-17B-16E-Instruct / critic=Llama-4-Scout-17B-16E-Instruct | 14 |